# SIC walkthrough: SuMoPP

A short tour of the `sic` package with the default **SuMoPP** (SuMo++) head and the two headline benchmark metrics:

1. Load METABRIC and preprocess one train/test split exactly like the benchmark.
2. Build the two evaluation grids: **unique test times** for `cindex_td` and the **q90 grid** for `ibs_q90`.
3. Predict on both grids, compute the metrics, and plot a few survival curves.
4. **Cross-validate** and save the per-fold metrics to CSV + JSON.

The whole benchmark (all SurvSet datasets + METABRIC, plus SEER / UNOS if available) runs with `bash examples/run_eval_survset.sh`.

Requirements: `pip install sic` (plus `matplotlib` for the plots). Model weights download automatically from [yayapa/sic](https://huggingface.co/yayapa/sic) on first use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sic import SIC, make_y, load_dataset, list_datasets
from sic.horizons import clip_times_to_interval, make_ibs_q90_grid, unique_sorted_times
from sic.metrics import cross_validate, evaluate_survival, prepare_fold, summarize

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
RANDOM_STATE = 42

## 1. Data and model

METABRIC is bundled with the package. `prepare_fold` is the benchmark preprocessing: median imputation and standardization of the numeric covariates, fitted on the training part only. `SIC()` uses the SuMoPP head; the checkpoint downloads on the first prediction. There is no gradient training at fit time: the training rows are consumed as in-context examples in a single forward pass.

In [ ]:
data = load_dataset("METABRIC")
idx_tr, idx_te = train_test_split(np.arange(len(data["x"])), test_size=0.3, random_state=RANDOM_STATE)
fold = prepare_fold(data["x"], data["t"], data["e"], data["num_cols"], idx_tr, idx_te)
x_tr, t_tr, e_tr = fold["train"]
x_te, t_te, e_te = fold["test"]
y_tr = make_y(t_tr, e_tr)

model = SIC()  # SuMoPP head; device defaults to CUDA when available
print(f"device={model.device}  head={model.survival_head}")
print(f"train={x_tr.shape}  test={x_te.shape}  event rate={e_tr.mean():.3f}")

## 2. The two evaluation grids

The metrics need the predicted curves on different time grids, and SIC is queried on each grid directly (no interpolation):

| metric | grid |
|---|---|
| `cindex_td` (time-dependent C-index, adjusted Antolini) | the unique **test** times |
| `ibs_q90` (integrated Brier score) | 100 points from the first **training** event time to the 90th percentile of the training durations, capped at the largest training time |

In [ ]:
cindex_grid = unique_sorted_times(t_te)
t_max_train = float(t_tr.max())
ibs_grid = clip_times_to_interval(
    make_ibs_q90_grid(t_tr, e_tr, percentile=90, n_points=100, max_time=t_max_train),
    None, t_max_train,
)
print(f"cindex_td grid: {len(cindex_grid)} times in [{cindex_grid[0]:.1f}, {cindex_grid[-1]:.1f}]")
print(f"ibs_q90 grid:   {len(ibs_grid)} times in [{ibs_grid[0]:.1f}, {ibs_grid[-1]:.1f}]")

## 3. Predict and evaluate

`evaluate_survival` takes the curve on the test-time grid plus the re-queried curve on the q90 grid (`surv_q90`) and returns the full metric suite; we show the two headline metrics.

In [ ]:
surv = model.fit_predict_surv(x_tr, y_tr, x_te, eval_times=cindex_grid.tolist())
surv_q90 = model.fit_predict_surv(x_tr, y_tr, x_te, eval_times=ibs_grid.tolist())
res = evaluate_survival(surv, t_te, e_te, t_tr, e_tr, surv_q90=surv_q90)

print(f"time scale (sumo_tscale_q, from the checkpoint config): {model.sumo_tscale_q}")
pd.Series({"cindex_td": res["cindex_td"], "ibs_q90": res["ibs_q90"]}, name="METABRIC test split").to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for j in range(5):
    ax.plot(ibs_grid, surv_q90.iloc[:, j], label=f"test subject {j}")
ax.set_xlabel("time"); ax.set_ylabel("survival probability"); ax.set_ylim(0, 1.02)
ax.set_title("SuMoPP survival curves on the ibs_q90 grid")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Cross-validation

`cross_validate` runs the same steps per fold (k-fold split, `prepare_fold`, both grids, plus the point-wise times for `auc@q` / `bs@q`) and returns one row per (dataset, fold). We use METABRIC here; `DATASETS = list_datasets()` runs the whole benchmark, which is better done with `bash examples/run_eval_survset.sh`.

In [ ]:
DATASETS = ["METABRIC"]  # list_datasets() for the whole benchmark
CSV_PATH = "sic_sumopp_cv.csv"
JSON_PATH = "sic_sumopp_cv.json"

results = cross_validate(
    model, DATASETS, k_folds=5, seed=RANDOM_STATE,
    csv_path=CSV_PATH, json_out=JSON_PATH,
)
print(f"saved {len(results)} rows to {CSV_PATH} and nested results to {JSON_PATH}")
summarize(results, metrics=("cindex_td", "ibs_q90"))